In [ ]:
from ultralytics import YOLO
import numpy as np

try:
    from ensemble_boxes import weighted_boxes_fusion
    USE_WBF = True
except ImportError:
    USE_WBF = False
    print("[INFO] ensemble-boxes not installed, fusion will use simple concatenation.")

# Load models
rgb_model = YOLO("yolov8_rgb.pt")
ir_model  = YOLO("yolov8_ir.pt")

def extract_boxes(results):
    boxes = []
    for r in results:
        for b in r.boxes:
            cls = int(b.cls.cpu().numpy())
            conf = float(b.conf.cpu().numpy())
            xywh = b.xywh.cpu().numpy()[0]  # [x_center, y_center, w, h]
            boxes.append([cls, conf, *xywh])
    return boxes

def to_wbf_format(boxes, img_w, img_h):
    b_list, s_list, l_list = [], [], []
    for cls, conf, xc, yc, w, h in boxes:
        x1 = (xc - w/2) / img_w
        y1 = (yc - h/2) / img_h
        x2 = (xc + w/2) / img_w
        y2 = (yc + h/2) / img_h
        b_list.append([x1, y1, x2, y2])
        s_list.append(conf)
        l_list.append(cls)
    return b_list, s_list, l_list

def late_fusion(rgb_img=None, ir_img=None):
    img_h, img_w = None, None
    fused = []

    if rgb_img is not None:
        rgb_results = rgb_model.predict(source=rgb_img, conf=0.25, verbose=False)
        rgb_boxes = extract_boxes(rgb_results)
        img_h, img_w = rgb_img.shape[:2]

    if ir_img is not None:
        ir_results = ir_model.predict(source=ir_img, conf=0.25, verbose=False)
        ir_boxes = extract_boxes(ir_results)
        if img_h is None:  # fallback if only IR
            img_h, img_w = ir_img.shape[:2]
    else:
        ir_boxes = []

    # Fusion logic
    if rgb_img is not None and ir_img is not None and USE_WBF:
        b_rgb, s_rgb, l_rgb = to_wbf_format(rgb_boxes, img_w, img_h)
        b_ir,  s_ir,  l_ir  = to_wbf_format(ir_boxes,  img_w, img_h)
        boxes, scores, labels = weighted_boxes_fusion(
            [b_rgb, b_ir], [s_rgb, s_ir], [l_rgb, l_ir],
            weights=[1,1], iou_thr=0.5, skip_box_thr=0.25
        )
        for b, s, l in zip(boxes, scores, labels):
            x1, y1, x2, y2 = b
            fused.append({
                "class": l,
                "conf": s,
                "bbox": [x1*img_w, y1*img_h, x2*img_w, y2*img_h]
            })
    else:
        # If only one modality or WBF not available
        for cls, conf, xc, yc, w, h in (rgb_boxes if rgb_img is not None else ir_boxes):
            fused.append({
                "class": cls,
                "conf": conf,
                "bbox": [(xc-w/2), (yc-h/2), (xc+w/2), (yc+h/2)]
            })

    return fused

## Real time

In [ ]:
import cv2
from ultralytics import YOLO
import numpy as np

try:
    from ensemble_boxes import weighted_boxes_fusion
    USE_WBF = True
except ImportError:
    USE_WBF = False

rgb_model = YOLO("yolov8_rgb.pt")
ir_model  = YOLO("yolov8_ir.pt")

def extract_boxes(results):
    boxes = []
    for r in results:
        for b in r.boxes:
            cls = int(b.cls.cpu().numpy())
            conf = float(b.conf.cpu().numpy())
            xyxy = b.xyxy.cpu().numpy()[0]
            boxes.append([cls, conf, *xyxy])
    return boxes

def fuse(rgb_boxes, ir_boxes, img_w, img_h):
    fused = []
    if rgb_boxes and ir_boxes and USE_WBF:
        def to_wbf(boxes):
            b_list, s_list, l_list = [], [], []
            for cls, conf, x1, y1, x2, y2 in boxes:
                b_list.append([x1/img_w, y1/img_h, x2/img_w, y2/img_h])
                s_list.append(conf)
                l_list.append(cls)
            return b_list, s_list, l_list
        b_rgb, s_rgb, l_rgb = to_wbf(rgb_boxes)
        b_ir,  s_ir,  l_ir  = to_wbf(ir_boxes)
        boxes, scores, labels = weighted_boxes_fusion(
            [b_rgb, b_ir], [s_rgb, s_ir], [l_rgb, l_ir],
            weights=[1,1], iou_thr=0.5, skip_box_thr=0.25
        )
        for (x1n,y1n,x2n,y2n),s,l in zip(boxes,scores,labels):
            fused.append([l,s,x1n*img_w,y1n*img_h,x2n*img_w,y2n*img_h])
    else:
        fused = rgb_boxes if rgb_boxes else ir_boxes
    return fused

cap_rgb = cv2.VideoCapture(0)   # camera 0 for RGB
cap_ir  = cv2.VideoCapture(1)    # camera 1 for IR

if not cap_rgb.isOpened() and not cap_ir.isOpened():
    print("[ERROR] No camera available.")
    exit()

while True:
    ret_rgb, frame_rgb = cap_rgb.read() if cap_rgb.isOpened() else (False,None)
    ret_ir, frame_ir   = cap_ir.read() if cap_ir.isOpened() else (False,None)

    rgb_boxes, ir_boxes = [], []
    if ret_rgb:
        rgb_results = rgb_model.predict(frame_rgb, conf=0.25, verbose=False)
        rgb_boxes = extract_boxes(rgb_results)
    if ret_ir:
        ir_results = ir_model.predict(frame_ir, conf=0.25, verbose=False)
        ir_boxes = extract_boxes(ir_results)

    display_frame = frame_rgb if ret_rgb else frame_ir
    if display_frame is None:
        break

    img_h,img_w = display_frame.shape[:2]
    fused = fuse(rgb_boxes, ir_boxes, img_w, img_h)

    for cls,conf,x1,y1,x2,y2 in fused:
        cv2.rectangle(display_frame,(int(x1),int(y1)),(int(x2),int(y2)),(0,255,0),2)
        cv2.putText(display_frame,f"drone {conf:.2f}",(int(x1),int(y1)-5),
                    cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)

    cv2.imshow("Late Fusion Detection", display_frame)
    if cv2.waitKey(1)&0xFF==ord('q'):
        break

cap_rgb.release()
cap_ir.release()
cv2.destroyAllWindows()